# DFU Repair-7 — GOOD38 Auto-Discover + CPU Repair
Auto-discovers the existing Drive run root by requiring its locked split manifest. Never creates a new run root; GOOD38 training remains forbidden.

In [ ]:
import json, urllib.request
from pathlib import Path

BASE_URL = "https://raw.githubusercontent.com/AzizulHakim00/DFU-ImageGuard/566c0f82ad557d188036d32f455bb5d5bdbb8538/notebooks/DFU_Repair7_GOOD38_UPLOAD_RECOVERY_CPU.ipynb"

raw = urllib.request.urlopen(BASE_URL, timeout=120).read()
nb = json.loads(raw.decode("utf-8"))
cells = [c for c in nb.get("cells", []) if c.get("cell_type") == "code"]
if len(cells) != 1:
    raise RuntimeError(f"Expected one code cell in pinned GOOD38 notebook, found {len(cells)}")
code = "".join(cells[0]["source"])

old_root_block = (
    'RUN_ID = "RELIABLE_DFU_CV_V3_MISSING38"\n'
    'RUN_ROOT = Path("/content/drive/MyDrive/DFU-ImageGuard/runs") / RUN_ID\n'
    'LOCKED_SPLIT = RUN_ROOT / "manifests" / "locked_outer_fold_assignments.csv"\n'
)
new_root_block = (
    'RUN_ID = "RELIABLE_DFU_CV_V3_MISSING38"\n'
    'RUN_ROOT = None\n'
    'LOCKED_SPLIT = None\n'
)
if code.count(old_root_block) != 1:
    raise RuntimeError(f"Root-definition patch target count != 1: {code.count(old_root_block)}")
code = code.replace(old_root_block, new_root_block, 1)

old_mount_block = (
    'from google.colab import drive, files\n'
    'if not Path("/content/drive/MyDrive").is_dir():\n'
    '    drive.mount("/content/drive")\n'
    'print("Google Drive mount: PASS")\n'
    '\n'
    'if not RUN_ROOT.is_dir():\n'
    '    raise RuntimeError(f"Existing run root not found: {RUN_ROOT}")\n'
    'if not LOCKED_SPLIT.is_file():\n'
    '    raise RuntimeError(f"Locked split not found: {LOCKED_SPLIT}")\n'
)

new_mount_block = r'''from google.colab import drive, files
if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")
print("Google Drive mount: PASS")

def _valid_run_root(p):
    p = Path(p)
    return (
        p.is_dir()
        and p.name == RUN_ID
        and (p / "manifests" / "locked_outer_fold_assignments.csv").is_file()
    )

def _discover_run_root():
    fast = [
        Path("/content/drive/MyDrive/DFU-ImageGuard/runs") / RUN_ID,
        Path("/content/drive/MyDrive/DFU_ImageGuard/runs") / RUN_ID,
        Path("/content/drive/MyDrive") / RUN_ID,
    ]
    found = []
    for p in fast:
        if _valid_run_root(p):
            found.append(p.resolve())

    if not found:
        for base in (Path("/content/drive/MyDrive"), Path("/content/drive/Shareddrives")):
            if not base.is_dir():
                continue
            try:
                for p in base.rglob(RUN_ID):
                    if _valid_run_root(p):
                        rp = p.resolve()
                        if rp not in found:
                            found.append(rp)
            except (PermissionError, OSError):
                pass

    uniq, seen = [], set()
    for p in found:
        s = str(p)
        if s not in seen:
            uniq.append(p)
            seen.add(s)

    if len(uniq) == 0:
        top = []
        for base in (Path("/content/drive/MyDrive"), Path("/content/drive/Shareddrives")):
            if base.is_dir():
                try:
                    top.extend(str(x) for x in list(base.iterdir())[:40])
                except Exception:
                    pass
        raise RuntimeError(
            "Could not auto-discover the existing DFU run. "
            "No RELIABLE_DFU_CV_V3_MISSING38 directory containing "
            "manifests/locked_outer_fold_assignments.csv was found. "
            "The wrong Google Drive account may be mounted, or the project folder was moved/deleted. "
            "Visible top-level entries: " + repr(top[:40])
        )

    if len(uniq) > 1:
        raise RuntimeError(
            "Multiple valid DFU run roots were found; refusing to guess: "
            + repr([str(x) for x in uniq])
        )
    return uniq[0]

RUN_ROOT = _discover_run_root()
LOCKED_SPLIT = RUN_ROOT / "manifests" / "locked_outer_fold_assignments.csv"
if RUN_ROOT.parent.name != "runs":
    raise RuntimeError(
        f"Discovered run root is not under a 'runs' directory: {RUN_ROOT}"
    )
DFU_DRIVE_ROOT = RUN_ROOT.parent.parent
print("Auto-discovered RUN_ROOT:", RUN_ROOT)
print("Auto-discovered DFU_DRIVE_ROOT:", DFU_DRIVE_ROOT)
print("Locked split:", LOCKED_SPLIT)
'''

if code.count(old_mount_block) != 1:
    raise RuntimeError(f"Mount/path patch target count != 1: {code.count(old_mount_block)}")
code = code.replace(old_mount_block, new_mount_block, 1)

old_launch = (
    'runner_code = "".join(cells[0]["source"])\n'
    'compile(runner_code, "DFU_Repair7_CPU_FINAL_FIXED_launcher.py", "exec")\n'
    'exec(compile(runner_code, "DFU_Repair7_CPU_FINAL_FIXED_launcher.py", "exec"), globals())\n'
)

new_launch = r'''runner_code = "".join(cells[0]["source"])

# Patch FINAL_FIXED so its loaded v1.7 source patches the direct v1.6 runner's
# hard-coded Drive roots before v1.6 can execute.
final_anchor = 'compile(code, "DFU_Repair7_CPU_FINAL_FIXED.py", "exec")\n'
if runner_code.count(final_anchor) != 1:
    raise RuntimeError(
        f"FINAL_FIXED injection anchor count != 1: {runner_code.count(final_anchor)}"
    )

drive_old = 'DRIVE_ROOT = Path("/content/drive/MyDrive/DFU-ImageGuard")'
drive_new = f'DRIVE_ROOT = Path({str(DFU_DRIVE_ROOT)!r})'
backup_old = 'BACKUP_ROOT = Path("/content/drive/MyDrive/DFU-ImageGuard-Backup")'
backup_path = DFU_DRIVE_ROOT.parent / (DFU_DRIVE_ROOT.name + "-Backup")
backup_new = f'BACKUP_ROOT = Path({str(backup_path)!r})'

direct_path_patch = (
    '\n# ---- auto-discovered Drive-root patch ----\n'
    f'_drive_old = {drive_old!r}\n'
    f'_drive_new = {drive_new!r}\n'
    f'_backup_old = {backup_old!r}\n'
    f'_backup_new = {backup_new!r}\n'
    'if script.count(_drive_old) != 1:\n'
    '    raise RuntimeError(f"direct DRIVE_ROOT target count != 1: {script.count(_drive_old)}")\n'
    'script = script.replace(_drive_old, _drive_new, 1)\n'
    'if script.count(_backup_old) != 1:\n'
    '    raise RuntimeError(f"direct BACKUP_ROOT target count != 1: {script.count(_backup_old)}")\n'
    'script = script.replace(_backup_old, _backup_new, 1)\n'
    'print("Direct runner Drive-root patch: PASS")\n'
    '# ---- end auto-discovered Drive-root patch ----\n'
)

v17_anchor = 'script = "".join(code_cells[0]["source"])\n'
final_injection = (
    '\n# Inject Drive-root patch into the v1.7 source before compilation.\n'
    f'_v17_anchor = {v17_anchor!r}\n'
    f'_direct_path_patch = {direct_path_patch!r}\n'
    'if code.count(_v17_anchor) != 1:\n'
    '    raise RuntimeError(f"v1.7 direct-runner anchor count != 1: {code.count(_v17_anchor)}")\n'
    'code = code.replace(_v17_anchor, _v17_anchor + _direct_path_patch, 1)\n'
    'print("Nested CPU runner path patch: PASS")\n'
)

runner_code = runner_code.replace(final_anchor, final_injection + '\n' + final_anchor, 1)
compile(runner_code, "DFU_Repair7_CPU_FINAL_FIXED_AUTOPATH_launcher.py", "exec")
print("CPU runner auto-path injection: PASS")
exec(
    compile(runner_code, "DFU_Repair7_CPU_FINAL_FIXED_AUTOPATH_launcher.py", "exec"),
    globals(),
)
'''

if code.count(old_launch) != 1:
    raise RuntimeError(f"Launch patch target count != 1: {code.count(old_launch)}")
code = code.replace(old_launch, new_launch, 1)

compile(code, "DFU_Repair7_GOOD38_AUTODISCOVER_CPU.py", "exec")
print("Generated patched base code compile: PASS")
